In [ ]:
import pandas as pd
from dask.diagnostics import ProgressBar
from pyproj import Transformer


In [1]:
from pystac_client import Client
import planetary_computer
from odc.stac import stac_load
import numpy as np
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import shutil

In [29]:
bbox = [77.90, 30.20, 78.20, 30.45]
month_range = "2023-11-01/2023-11-30"
year_range = "2023-01-01/2023-12-31"


In [30]:
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

In [16]:
print("Searching Sentinel scenes...")

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=month_range,
    query={"eo:cloud_cover": {"lt": 10}},
)

items = list(search.get_items())
print("Scenes found:", len(items))

items = [planetary_computer.sign(item) for item in items]

ds = stac_load(
    items,
    bands=["B02","B03","B04","B08"],
    bbox=bbox,
    resolution=10,
    chunks={"x":1024,"y":1024},
)

print("Downloading Sentinel data...")

with ProgressBar():
    ds = ds.load()

print("Sentinel loaded.")
print(ds)

Searching Sentinel scenes...
Scenes found: 12
[########################################] | 100% Completed | 233.26 s
Sentinel loaded.
<xarray.Dataset> Size: 539MB
Dimensions:      (y: 2848, x: 2959, time: 4)
Coordinates:
  * y            (y) float64 23kB 3.373e+06 3.373e+06 ... 3.344e+06 3.344e+06
  * x            (x) float64 24kB 2.015e+05 2.016e+05 ... 2.311e+05 2.311e+05
  * time         (time) datetime64[ns] 32B 2023-11-05T05:30:11.024000 ... 202...
    spatial_ref  int32 4B 32644
Data variables:
    B02          (time, y, x) float32 135MB 1.227e+03 1.234e+03 ... 1.38e+03
    B03          (time, y, x) float32 135MB 1.413e+03 1.467e+03 ... 1.47e+03
    B04          (time, y, x) float32 135MB 1.298e+03 1.31e+03 ... 1.547e+03
    B08          (time, y, x) float32 135MB 2.992e+03 3.348e+03 ... 2.526e+03


In [17]:
print("Loading IO LULC...")

search_lulc = catalog.search(
    collections=["io-lulc-annual-v02"],
    bbox=bbox,
    datetime=year_range,
)

lulc_items = list(search_lulc.get_items())
lulc_items = [planetary_computer.sign(item) for item in lulc_items]

lulc_ds = stac_load(
    lulc_items,
    bbox=bbox,
    resolution=10,
    chunks={"x":1024,"y":1024},
)

with ProgressBar():
    lulc_ds = lulc_ds.load()

lulc = lulc_ds.to_array().values[0].astype(np.int64)

print("LULC shape:", lulc.shape)

Loading IO LULC...
[########################################] | 100% Completed | 5.79 sms
LULC shape: (2, 2848, 2959)


In [18]:
transformer = Transformer.from_crs(ds.odc.crs, "EPSG:4326", always_xy=True)

def normalize_time(dt):
    week = dt.timetuple().tm_yday / 365.0
    hour = dt.hour / 24.0
    return week, hour

def normalize_latlon(lat, lon):
    lat_norm = (lat + 90) / 180.0
    lon_norm = (lon + 180) / 360.0
    return lat_norm, lon_norm

def encode_scalar(x):
    return np.array([
        np.sin(2*np.pi*x),
        np.cos(2*np.pi*x)
    ], dtype=np.float32)

In [ ]:
CHIP_SIZE = 256
chips = []
masks = []
latlon_meta = []
time_meta = []

for t_idx in tqdm(range(len(ds.time)), desc="Processing timesteps"):

    dt = pd.to_datetime(ds.time.values[t_idx]).to_pydatetime()
    week_norm, hour_norm = normalize_time(dt)

    week_enc = encode_scalar(week_norm)
    hour_enc = encode_scalar(hour_norm)

    img = ds.isel(time=t_idx).to_array().values.astype(np.float32) / 10000.0
    _, H, W = img.shape

    for i in range(0, H - CHIP_SIZE, CHIP_SIZE):
        for j in range(0, W - CHIP_SIZE, CHIP_SIZE):

            chip = img[:, i:i+CHIP_SIZE, j:j+CHIP_SIZE]
            mask_chip = lulc[i:i+CHIP_SIZE, j:j+CHIP_SIZE]

            if mask_chip.ndim != 2:
                print("Mask dimension error:", mask_chip.shape)
                continue
            if chip.shape != (4, CHIP_SIZE, CHIP_SIZE):
                continue

            # center pixel coordinate
            y_coord = float(ds.y.values[i + CHIP_SIZE//2])
            x_coord = float(ds.x.values[j + CHIP_SIZE//2])

            lon, lat = transformer.transform(x_coord, y_coord)

            lat_norm, lon_norm = normalize_latlon(lat, lon)
            lat_enc = encode_scalar(lat_norm)
            lon_enc = encode_scalar(lon_norm)

            chips.append(chip)
            masks.append(mask_chip)
            latlon_meta.append(np.concatenate([lat_enc, lon_enc]))
            time_meta.append(np.concatenate([week_enc, hour_enc]))

print("Total chips created:", len(chips))

Processing timesteps: 100%|██████████| 4/4 [00:00<00:00,  5.71it/s]

Total chips created: 484


In [20]:
os.makedirs("cubes", exist_ok=True)

for idx in tqdm(range(len(chips))):

    np.savez(
        f"cubes/cube_{idx}.npz",
        image=chips[idx].astype(np.float32),      # (4,128,128)
        mask=masks[idx].astype(np.int64),         # (128,128)
        latlon=latlon_meta[idx],                 # (4,)
        time=time_meta[idx]                      # (4,)
    )

100%|██████████| 484/484 [00:01<00:00, 278.60it/s]


In [21]:
print(ds.odc.geobox)
print(lulc_ds.odc.geobox)

GeoBox((2848, 2959), Affine(10.0, 0.0, 201540.0,
       0.0, -10.0, 3372740.0), CRS('PROJCRS["WGS 84 / UTM zone 44N",BASEGEOGCRS["WGS 84",ENSEMBLE["World Geodetic System 1984 ensemble",MEMBER["World Geodetic System 1984 (Transit)"],MEMBER["World Geodetic System 1984 (G730)"],MEMBER["World Geodetic System 1984 (G873)"],MEMBER["World Geodetic System 1984 (G1150)"],MEMBER["World Geodetic System 1984 (G1674)"],MEMBER["World Geodetic System 1984 (G1762)"],MEMBER["World Geodetic System 1984 (G2139)"],MEMBER["World Geodetic System 1984 (G2296)"],ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]],ENSEMBLEACCURACY[2.0]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]],ID["EPSG",4326]],CONVERSION["UTM zone 44N",METHOD["Transverse Mercator",ID["EPSG",9807]],PARAMETER["Latitude of natural origin",0,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8801]],PARAMETER["Longitude of natural origin",81,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8802]],PARAMETER["Scale facto

In [22]:
lulc = lulc_ds.to_array().values[0]

In [24]:
print(lulc_ds)
print(lulc_ds.dims)

<xarray.Dataset> Size: 67MB
Dimensions:      (y: 2848, x: 2959, time: 2)
Coordinates:
  * y            (y) float64 23kB 3.373e+06 3.373e+06 ... 3.344e+06 3.344e+06
  * x            (x) float64 24kB 2.015e+05 2.016e+05 ... 2.311e+05 2.311e+05
  * time         (time) datetime64[ns] 16B 2022-01-01 2023-01-01
    spatial_ref  int32 4B 32644
Data variables:
    data         (time, y, x) float32 67MB 2.0 2.0 2.0 2.0 ... 2.0 2.0 11.0 11.0
FrozenMappingWarningOnValuesAccess({'y': 2848, 'x': 2959, 'time': 2})


In [26]:
print(ds)

<xarray.Dataset> Size: 539MB
Dimensions:      (y: 2848, x: 2959, time: 4)
Coordinates:
  * y            (y) float64 23kB 3.373e+06 3.373e+06 ... 3.344e+06 3.344e+06
  * x            (x) float64 24kB 2.015e+05 2.016e+05 ... 2.311e+05 2.311e+05
  * time         (time) datetime64[ns] 32B 2023-11-05T05:30:11.024000 ... 202...
    spatial_ref  int32 4B 32644
Data variables:
    B02          (time, y, x) float32 135MB 1.227e+03 1.234e+03 ... 1.38e+03
    B03          (time, y, x) float32 135MB 1.413e+03 1.467e+03 ... 1.47e+03
    B04          (time, y, x) float32 135MB 1.298e+03 1.31e+03 ... 1.547e+03
    B08          (time, y, x) float32 135MB 2.992e+03 3.348e+03 ... 2.526e+03


In [27]:
print(ds.time.values)

['2023-11-05T05:30:11.024000000' '2023-11-15T05:31:01.024000000'
 '2023-11-20T05:31:09.024000000' '2023-11-25T05:31:41.024000000']


In [28]:
# Select 2023 LULC explicitly
lulc_2023 = lulc_ds.sel(time="2023-01-01")

# Extract only spatial array
lulc = lulc_2023["data"].values.astype(np.int64)

print("Final LULC shape:", lulc.shape)

Final LULC shape: (2848, 2959)
